In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_style("whitegrid")

# ============================================================
# LOAD PROCESSED DATA
# ============================================================

df = pd.read_pickle('/Users/gihbeom/Desktop/Data Science Preparation/fqhc-chronic-disease-analysis/data/processed/hc2024_processed.pkl')

print("="*80)
print("BUSINESS IMPACT ANALYSIS — FQHC CHRONIC DISEASE MANAGEMENT")
print("="*80)
print()

# ============================================================
# FQHC SECTOR CONTEXT
# ============================================================

print("NATIONAL FQHC SECTOR OVERVIEW")
print("-"*80)

# Constants (2024 data)
NUM_FQHCS_US = 1400  # Number of FQHCs in US
ANNUAL_PATIENTS = 30_000_000  # Total patients served by FQHCs
FEDERAL_FUNDING = 6_000_000_000  # Federal HRSA funding ($6B)
AVG_VISIT_COST = 250  # Average cost per FQHC visit

print(f"FQHCs in United States: {NUM_FQHCS_US:,}")
print(f"Annual patients served: {ANNUAL_PATIENTS:,}")
print(f"Federal funding (2024): ${FEDERAL_FUNDING:,.0f}")
print(f"Average visit cost: ${AVG_VISIT_COST}")

# Calculate national visit volume from our weighted data
national_visits = df['VISWT'].sum()
print(f"\nWeighted national FQHC visits (from sample): {national_visits:,.0f}")

# ============================================================
# CHRONIC DISEASE BURDEN QUANTIFICATION
# ============================================================

print("\n" + "="*80)
print("CHRONIC DISEASE BURDEN")
print("="*80)

chronic_conditions = ['diabetes', 'hypertension', 'mental_health', 'obesity', 
                     'asthma_copd', 'chd', 'ckd', 'substance_use']

# Calculate prevalence and national estimates
burden_data = []
for condition in chronic_conditions:
    prevalence = df[condition].mean()
    national_cases = national_visits * prevalence
    
    burden_data.append({
        'Condition': condition.replace('_', ' ').title(),
        'Prevalence (%)': prevalence * 100,
        'National Visit Volume': int(national_cases)
    })

burden_df = pd.DataFrame(burden_data).sort_values('National Visit Volume', ascending=False)
print("\n" + burden_df.to_string(index=False))

# ============================================================
# ROI SCENARIO 1: DIABETES EARLY INTERVENTION
# ============================================================

print("\n" + "="*80)
print("ROI SCENARIO 1: DIABETES EARLY INTERVENTION PROGRAM")
print("="*80)

# Parameters
diabetes_visits_national = national_visits * df['diabetes'].mean()
complication_rate_baseline = 0.20  # 20% develop complications without intervention
complication_cost = 12000  # Cost per diabetes complication (ER visit, hospitalization)
intervention_reduction = 0.30  # 30% reduction in complications with early intervention
cost_per_intervention = 150  # Cost of diabetes education/monitoring program

# Calculations
baseline_complications = diabetes_visits_national * complication_rate_baseline
baseline_complication_cost = baseline_complications * complication_cost

with_intervention_complications = baseline_complications * (1 - intervention_reduction)
with_intervention_complication_cost = with_intervention_complications * complication_cost

intervention_program_cost = diabetes_visits_national * cost_per_intervention

# Savings
avoided_complications = baseline_complications - with_intervention_complications
complication_cost_savings = baseline_complication_cost - with_intervention_complication_cost
net_savings_diabetes = complication_cost_savings - intervention_program_cost

print(f"\nDiabetes visits nationally: {diabetes_visits_national:,.0f}")
print(f"\nBaseline (No Intervention):")
print(f"  Expected complications: {baseline_complications:,.0f}")
print(f"  Complication costs: ${baseline_complication_cost:,.0f}")

print(f"\nWith Early Intervention Program:")
print(f"  Reduced complications: {with_intervention_complications:,.0f}")
print(f"  Complication costs: ${with_intervention_complication_cost:,.0f}")
print(f"  Program cost: ${intervention_program_cost:,.0f}")

print(f"\n📊 NET IMPACT:")
print(f"  Complications avoided: {avoided_complications:,.0f}")
print(f"  Cost savings: ${complication_cost_savings:,.0f}")
print(f"  Program cost: ${intervention_program_cost:,.0f}")
print(f"  💰 NET SAVINGS: ${net_savings_diabetes:,.0f}")

# ============================================================
# ROI SCENARIO 2: HYPERTENSION CONTROL PROGRAM
# ============================================================

print("\n" + "="*80)
print("ROI SCENARIO 2: HYPERTENSION CONTROL PROGRAM")
print("="*80)

# Parameters
htn_visits_national = national_visits * df['hypertension'].mean()
uncontrolled_rate = 0.45  # 45% have uncontrolled hypertension
cvd_risk_reduction = 0.25  # 25% reduction in cardiovascular events
cvd_event_cost = 28000  # Cost per cardiovascular event
cvd_event_rate = 0.08  # 8% annual CVD event rate among uncontrolled HTN
monitoring_cost_per_visit = 75  # Cost of BP monitoring program

# Calculations
uncontrolled_htn_visits = htn_visits_national * uncontrolled_rate
baseline_cvd_events = uncontrolled_htn_visits * cvd_event_rate
baseline_cvd_cost = baseline_cvd_events * cvd_event_cost

with_program_cvd_events = baseline_cvd_events * (1 - cvd_risk_reduction)
with_program_cvd_cost = with_program_cvd_events * cvd_event_cost

program_cost_htn = htn_visits_national * monitoring_cost_per_visit

# Savings
cvd_events_avoided = baseline_cvd_events - with_program_cvd_events
cvd_cost_savings = baseline_cvd_cost - with_program_cvd_cost
net_savings_htn = cvd_cost_savings - program_cost_htn

print(f"\nHypertension visits nationally: {htn_visits_national:,.0f}")
print(f"Uncontrolled HTN visits: {uncontrolled_htn_visits:,.0f}")

print(f"\nBaseline (No Program):")
print(f"  Expected CVD events: {baseline_cvd_events:,.0f}")
print(f"  CVD event costs: ${baseline_cvd_cost:,.0f}")

print(f"\nWith BP Control Program:")
print(f"  Reduced CVD events: {with_program_cvd_events:,.0f}")
print(f"  CVD event costs: ${with_program_cvd_cost:,.0f}")
print(f"  Program cost: ${program_cost_htn:,.0f}")

print(f"\n📊 NET IMPACT:")
print(f"  CVD events avoided: {cvd_events_avoided:,.0f}")
print(f"  Cost savings: ${cvd_cost_savings:,.0f}")
print(f"  Program cost: ${program_cost_htn:,.0f}")
print(f"  💰 NET SAVINGS: ${net_savings_htn:,.0f}")

# ============================================================
# ROI SCENARIO 3: MENTAL HEALTH INTEGRATION
# ============================================================

print("\n" + "="*80)
print("ROI SCENARIO 3: INTEGRATED MENTAL HEALTH SERVICES")
print("="*80)

# Parameters
mh_visits_national = national_visits * df['mental_health'].mean()
untreated_rate = 0.60  # 60% not receiving adequate MH treatment
crisis_rate = 0.12  # 12% experience mental health crisis
crisis_cost = 8500  # Cost per crisis (ER visit, crisis intervention)
treatment_effectiveness = 0.40  # 40% reduction in crises with integrated care
integrated_care_cost = 200  # Cost per visit for integrated MH services

# Calculations
untreated_mh_visits = mh_visits_national * untreated_rate
baseline_crises = untreated_mh_visits * crisis_rate
baseline_crisis_cost = baseline_crises * crisis_cost

with_integrated_crises = baseline_crises * (1 - treatment_effectiveness)
with_integrated_crisis_cost = with_integrated_crises * crisis_cost

program_cost_mh = mh_visits_national * integrated_care_cost

# Savings
crises_avoided = baseline_crises - with_integrated_crises
crisis_cost_savings = baseline_crisis_cost - with_integrated_crisis_cost
net_savings_mh = crisis_cost_savings - program_cost_mh

print(f"\nMental health visits nationally: {mh_visits_national:,.0f}")
print(f"Untreated/undertreated: {untreated_mh_visits:,.0f}")

print(f"\nBaseline (No Integrated Care):")
print(f"  Expected MH crises: {baseline_crises:,.0f}")
print(f"  Crisis costs: ${baseline_crisis_cost:,.0f}")

print(f"\nWith Integrated MH Services:")
print(f"  Reduced crises: {with_integrated_crises:,.0f}")
print(f"  Crisis costs: ${with_integrated_crisis_cost:,.0f}")
print(f"  Program cost: ${program_cost_mh:,.0f}")

print(f"\n📊 NET IMPACT:")
print(f"  Crises avoided: {crises_avoided:,.0f}")
print(f"  Cost savings: ${crisis_cost_savings:,.0f}")
print(f"  Program cost: ${program_cost_mh:,.0f}")
print(f"  💰 NET SAVINGS: ${net_savings_mh:,.0f}")

# ============================================================
# TOTAL NATIONAL IMPACT
# ============================================================

print("\n" + "="*80)
print("TOTAL NATIONAL IMPACT — ALL THREE PROGRAMS")
print("="*80)

total_savings = net_savings_diabetes + net_savings_htn + net_savings_mh
total_program_cost = intervention_program_cost + program_cost_htn + program_cost_mh

print(f"\nProgram Savings:")
print(f"  Diabetes Early Intervention:     ${net_savings_diabetes:>15,.0f}")
print(f"  Hypertension Control:             ${net_savings_htn:>15,.0f}")
print(f"  Integrated Mental Health:         ${net_savings_mh:>15,.0f}")
print(f"  {'-'*50}")
print(f"  💰 TOTAL NET SAVINGS:             ${total_savings:>15,.0f}")

print(f"\nROI Metrics:")
roi_ratio = total_savings / total_program_cost
print(f"  Total program investment:         ${total_program_cost:>15,.0f}")
print(f"  Total savings generated:          ${total_savings:>15,.0f}")
print(f"  📈 ROI RATIO:                      {roi_ratio:>15.2f}x")
print(f"  💵 RETURN:                         ${roi_ratio:.0f} saved for every $1 invested")

# ============================================================
# VISUALIZATION: ROI WATERFALL
# ============================================================

print("\nCreating ROI Waterfall Visualization...")

fig, ax = plt.subplots(figsize=(12, 7))

categories = ['Diabetes\nIntervention', 'Hypertension\nControl', 
              'Mental Health\nIntegration', 'Total\nSavings']
values = [net_savings_diabetes/1e6, net_savings_htn/1e6, 
          net_savings_mh/1e6, total_savings/1e6]

colors = ['#2ecc71', '#3498db', '#9b59b6', '#e74c3c']

bars = ax.bar(range(len(categories)), values, color=colors, alpha=0.8, edgecolor='black', linewidth=2)

# Add value labels
for i, v in enumerate(values):
    ax.text(i, v + 5, f'${v:.0f}M', ha='center', fontsize=12, fontweight='bold')

ax.set_ylabel('Net Savings ($ Millions)', fontsize=13, fontweight='bold')
ax.set_title('National Savings Potential from FQHC Chronic Disease Programs', 
             fontsize=14, fontweight='bold', pad=20)
ax.set_xticks(range(len(categories)))
ax.set_xticklabels(categories, fontsize=11, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0, max(values) + 20)

# Add annotation
ax.text(0.98, 0.95, f'ROI: ${roi_ratio:.1f} per $1 invested', 
        transform=ax.transAxes, fontsize=12, fontweight='bold',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8),
        verticalalignment='top', horizontalalignment='right')

plt.tight_layout()
plt.savefig('/Users/gihbeom/Desktop/Data Science Preparation/fqhc-chronic-disease-analysis/figures/06_roi_waterfall.png', 
            dpi=300, bbox_inches='tight')
print("✓ Saved: figures/06_roi_waterfall.png")
plt.close()

# ============================================================
# SAVE ROI SUMMARY
# ============================================================

roi_summary = pd.DataFrame({
    'Program': [
        'Diabetes Early Intervention',
        'Hypertension Control',
        'Integrated Mental Health',
        'TOTAL'
    ],
    'Program Cost': [
        f'${intervention_program_cost:,.0f}',
        f'${program_cost_htn:,.0f}',
        f'${program_cost_mh:,.0f}',
        f'${total_program_cost:,.0f}'
    ],
    'Net Savings': [
        f'${net_savings_diabetes:,.0f}',
        f'${net_savings_htn:,.0f}',
        f'${net_savings_mh:,.0f}',
        f'${total_savings:,.0f}'
    ],
    'ROI Ratio': [
        f'{net_savings_diabetes/intervention_program_cost:.2f}x',
        f'{net_savings_htn/program_cost_htn:.2f}x',
        f'{net_savings_mh/program_cost_mh:.2f}x',
        f'{roi_ratio:.2f}x'
    ]
})

print("\n" + "="*80)
print("ROI SUMMARY TABLE")
print("="*80)
print("\n" + roi_summary.to_string(index=False))

roi_summary.to_csv('/Users/gihbeom/Desktop/Data Science Preparation/fqhc-chronic-disease-analysis/outputs/roi_analysis.csv', 
                   index=False)
print("\n✓ Saved: outputs/roi_analysis.csv")

print("\n" + "="*80)
print("BUSINESS IMPACT ANALYSIS COMPLETE")
print("="*80)
print(f"✓ Total national savings potential: ${total_savings:,.0f}")
print(f"✓ ROI ratio: {roi_ratio:.1f}x")
print(f"✓ Ready for final documentation")

BUSINESS IMPACT ANALYSIS — FQHC CHRONIC DISEASE MANAGEMENT

NATIONAL FQHC SECTOR OVERVIEW
--------------------------------------------------------------------------------
FQHCs in United States: 1,400
Annual patients served: 30,000,000
Federal funding (2024): $6,000,000,000
Average visit cost: $250

Weighted national FQHC visits (from sample): 123,817,677

CHRONIC DISEASE BURDEN

    Condition  Prevalence (%)  National Visit Volume
Mental Health       18.351168               22721989
 Hypertension       15.425993               19100106
      Obesity       12.599668               15600615
     Diabetes       10.358695               12825895
Substance Use        6.836258                8464495
  Asthma Copd        5.394612                6679482
          Ckd        1.792183                2219039
          Chd        1.538312                1904702

ROI SCENARIO 1: DIABETES EARLY INTERVENTION PROGRAM

Diabetes visits nationally: 12,825,895

Baseline (No Intervention):
  Expected complic